# Comp1: Conventional radiomics

This notebook demonstrates conventional radiomics modeling and characterization, including the potential contribution of a radiomics score to clinical diagnosis.

Typical input data (file and directory names may vary):
1. `images`: CT or MRI images for all study participants.
2. `masks`: manually delineated regions of interest, matched to the image filenames.
3. `label.txt`: patient-level labels, such as tumor malignancy or five-year survival status.

## Onekey workflow

1. Validate the input data format.
2. Extract radiomics features after validation.
3. Load the outcome labels.
4. Merge features and labels into an analysis dataset.
5. Inspect descriptive statistics and potential outliers.
6. Standardize features to zero mean and unit variance.
7. Filter features using correlation coefficients, such as Spearman or Pearson correlation.
8. Prepare training and test cohorts; adapt cohort definitions for multicenter validation.
9. Select nonzero features using LASSO (Cox-LASSO for survival analysis).
10. Fit the model appropriate to the outcome; this notebook uses Cox regression.
11. Visualize model performance and survival outcomes.


In [ ]:
## Video tutorials
import os
os.environ['DISABLE_VIDEO'] = 'TRUE'
from onekey_algo.custom.Manager import onekey_show
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1')

## 1. Data validation

Check the diagnostic messages before continuing. If validation fails, adjust the data according to the reported issues.

**Note:** image and mask filenames must match exactly. For example, `images/1.nii.gz` must have a corresponding `masks/1.nii.gz`.

Custom functions can also be used to locate and parse the data.

In [ ]:
# Data validation tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u6570\u636e\u68c0\u9a8c')

### Specify the input data

Define these three parameters:
1. `mydir`: path to the data directory.
2. `labelf`: file containing sample-level labels.
3. `labels`: target outcomes, such as tumor malignancy or T stage.

In [ ]:
import os
import pandas as pd
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('features', exist_ok=True)

# Set the task prefix
task_type = 'Peri5mm_'
# Set the data directory
# mydir = r'/path/to/your/data'
mydir = get_param_in_cwd('radio_dir')
# Corresponding label file
group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file')
# Read the outcome column names
labels = [get_param_in_cwd('task_column') or 'label']

### Match images and masks

Image and mask filenames must match exactly. For example, `images/1.nii.gz` requires a corresponding `masks/1.nii.gz`.

Custom functions may also be used to locate and parse the data.

In [ ]:
from pathlib import Path
from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir

for modal in get_param_in_cwd('modals'):
    images, masks = get_image_mask_from_dir(os.path.join(mydir), images=f'images_re', masks=f'masks_p{task_type[-4]}')
    # diagnose_3d_image_mask_settings(images, masks, verbose=False)
    print(f'Modality: {modal}, found {len(images)} samples.')

# Conventional radiomics features

Extract features using PyRadiomics through the Onekey interfaces below. This section usually requires no changes.

```python
def extract(self, images: Union[str, List[str]],
            masks: Union[str, List[str]], labels: Union[int, List[int]] = 1, settings=None)
```

- `images`: list of images to process.
- `masks`: corresponding masks, matched one-to-one with `images`.
- `labels`: mask labels from which to extract features; defaults to 1.
- `settings`: additional extraction settings; defaults to `None`.

```python
def get_label_data_frame(self, label: int = 1, column_names=None, images='images', masks='labels')
```

- `label`: mask label whose features are returned.
- `column_names`: optional column names; `None` uses the default names.

```python
def get_image_mask_from_dir(root, images='images', masks='labels')
```

- `root`: directory containing the input data.
- `images`: image subdirectory within `root`.
- `masks`: mask subdirectory within `root`.


In [ ]:
import warnings
import pandas as pd
import numpy as np
from onekey_algo.custom.components.Radiology import ConventionalRadiomics
from onekey_algo.custom.components.habitat import habitat_feature_fusion
 
warnings.filterwarnings("ignore")

rad_data = None
for modal in get_param_in_cwd('modals'):
    if os.path.exists(f'features/rad_features_{modal}_{task_type.strip("_")}.csv'):
        rad_data_ = pd.read_csv(f'features/rad_features_{modal}_{task_type.strip("_")}.csv', header=0)
    else:
        images, masks = get_image_mask_from_dir(os.path.join(mydir), images=f'images_re', masks=f'masks_p{task_type[-4]}')
        param_file = rf'./custom_settings/exampleCT.yaml'
        radiomics = ConventionalRadiomics(param_file, correctMask=True)
        radiomics.extract(images, masks, workers=6, labels=2)
        rad_data_ = radiomics.get_label_data_frame(label=2)
        rad_data_.columns = [f"{c.replace('-', '_')}" if c != 'ID' else 'ID' for c in rad_data_.columns]
        rad_data_.to_csv(f'features/rad_features_{modal}_{task_type.strip("_")}.csv', header=True, index=False)
    
    # rad_data_.columns = ['ID'] + [f"{c}_{modal}" for c in rad_data_.columns if c != 'ID']
    if rad_data is None:
        rad_data = rad_data_
    else:
        rad_data = rad_data.merge(rad_data_, on='ID', how='inner')
# rad_data = rad_data[0]
# rad_data = habitat_feature_fusion(*rad_data, mode='fill')
rad_data

## Feature statistics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sorted_counts = pd.DataFrame([c.split('_')[-2] for c in rad_data.columns if c !='ID']).value_counts()
sorted_counts = pd.DataFrame(sorted_counts, columns=['count']).reset_index()
sorted_counts = sorted_counts.sort_values(0)
sorted_counts.columns = ['feature_group', 'count']
display(sorted_counts)

plt.figure(figsize=(20, 10))
ax = plt.subplot(121)
plt.pie(sorted_counts['count'], labels=[i for i in sorted_counts['feature_group']], startangle=0,
        counterclock = False, autopct = '%.1f%%')
# plt.bar_label(bar.containers[0])
ax = plt.subplot(122)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
bar = sns.barplot(data=sorted_counts, x='feature_group', y='count', )
plt.savefig(f'img/{task_type}feature_ratio.svg', bbox_inches = 'tight')

## Outcome labels

Labels are stored in CSV format. For other formats, use a custom reader.

`label_data` must be a `DataFrame` with an `ID` column and one or more outcome columns. Multiple columns support multiple tasks.

In [ ]:
from onekey_algo.custom.components.comp1 import fillna
label_data = pd.read_csv(get_param_in_cwd('label_file'))
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# label_data = fillna(label_data, fill_mod='50%')
label_data

## Merge features and labels

Merge `label_data` with `rad_data` to construct the analysis dataset.

**Notes:**
1. Exclude `ID` from the predictor matrix.
2. If samples are missing after merging, check that their identifiers match.

In [ ]:
from onekey_algo.custom.utils import print_join_info
event_col = get_param_in_cwd('event_col', 'event')
duration_col= get_param_in_cwd('duration_col', 'time')

print_join_info(rad_data, label_data)
combined_data = pd.merge(rad_data, label_data[['ID', event_col, duration_col, 'group']], on=['ID'], how='inner')
print(combined_data[event_col].value_counts())
combined_data

## Descriptive statistics

1. `count`: number of samples.
2. `mean`, `std`: feature mean and standard deviation.
3. `min`, `25%`, `50%`, `75%`, `max`: minimum, quartiles, median, and maximum.

In [ ]:
combined_data.describe()

## Standardization

Onekey's `normalize_df` standardizes features to zero mean and unit variance:

$column = \frac{column - mean}{std}$

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df

data = normalize_df(combined_data, not_norm=['ID', event_col, duration_col], group='group', use_train=True)
data = data.dropna(axis=1)
data.to_csv(f'features/{task_type}feature_norm.csv', index=False)
data.describe()

### Correlation coefficients

Three methods are available:
1. `pearson`: Pearson correlation coefficient.
2. `kendall`: Kendall tau correlation coefficient.
3. `spearman`: Spearman rank correlation coefficient.

Reference: https://blog.csdn.net/zmqsdu9001/article/details/82840332

In [ ]:
# Correlation tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u76f8\u5173\u7cfb\u6570')

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df, select_feature

corr_name = get_param_in_cwd('corr_name', 'pearson')
if os.path.exists(f'features/{task_type}rad_features_corrsel.csv') and False:
    data = pd.read_csv(f'features/{task_type}rad_features_corrsel.csv', header=0)
else:
    tgroup = data[data['group'] == 'train']
    sel_feature = select_feature(tgroup[[c for c in tgroup.columns if c not in [event_col, duration_col]]].corr(corr_name), 
                                 threshold=0.9, topn=2048, verbose=False)
    data = data[['ID'] + sel_feature + [event_col, duration_col, 'group']]
    data.to_csv(f'features/{task_type}rad_features_corrsel.csv', header=True, index=False)
data

## Prepare the cohorts

Separate predictors `X` from outcomes `y` and prepare training and evaluation datasets. A common generic split is 80:20; the code below uses the cohort assignments supplied in the input data.

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict

train_data = data[(data[group_info] == 'train')]

# subsets = [s for s in label_data['group'].value_counts().index if s != 'train']
subsets = get_param_in_cwd('subsets')
val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_features_norm.csv', index=False)

print(', '.join([f"{subset} sample dimensions: {d_.shape}" for subset, d_ in val_datasets.items()]))

# Univariable Cox regression

In [ ]:
from onekey_algo.custom.components.survival import uni_cox

if os.path.exists(f'features/{task_type}rad_features_unisel.csv') and False:
    train_data = pd.read_csv(f'features/{task_type}rad_features_unisel.csv')
else:
    sel_features = uni_cox(train_data, duration_col=duration_col, event_col=event_col,
                           cols=[c for c in train_data.columns if c not in [event_col, duration_col, 'ID', 'group']], 
                           verbose=False)
    train_data = train_data[['ID'] + sel_features + [event_col, duration_col, 'group']]
    train_data.to_csv(f'features/{task_type}rad_features_unisel.csv', header=True, index=False)
train_data

# Cox-LASSO feature selection

In [ ]:
from onekey_algo.custom.components.survival import get_x_y_survival, lasso_cox_cv
COEF_THRESHOLD = 1e-6

X, y = get_x_y_survival(train_data, val_outcome=1, event_col=event_col, duration_col=duration_col)
sel_features = lasso_cox_cv(X, y, max_iter=100,  norm_X=False, prefix=f"{task_type}", l1_ratio=0.1, cv=10, weights_fig_size=(10, 10))
# sel_features = lasso_cox_cv(X, y, max_iter=1000,  norm_X=False, prefix=f"{task}_", l1_ratio=0.8, cv=10)

# Selected features

Use the features retained by Cox-LASSO.

In [ ]:
train_data = train_data[['ID'] + list(sel_features.index) + [event_col, duration_col]]
for subset in subsets:
    val_datasets[subset] = val_datasets[subset][['ID'] + list(sel_features.index) + [event_col, duration_col]]
    val_datasets[subset].to_csv(f'features/{task_type}{subset}_cox.csv', index=False)
rad_data[['ID'] + list(sel_features.index)].to_csv(f'features/{task_type}cox_sel.csv', index=False)

# Cluster analysis of selected features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if train_data.shape[1] < 150:
    pp = sns.clustermap(train_data[[c for c in train_data.columns if c not in [event_col, duration_col]]].corr(corr_name), 
                        linewidths=.5, figsize=(20.0, 16.0), cmap='YlGnBu')
    plt.setp(pp.ax_heatmap.get_yticklabels(), rotation=0)
    plt.savefig(f'img/{task_type}feature_cluster.svg', bbox_inches = 'tight')

# Cox regression modeling

In [ ]:
from lifelines import CoxPHFitter
mci = 0
max_p = 0

cph = CoxPHFitter(penalizer=max_p/100)
cph.fit(train_data[[c for c in train_data.columns if c != 'ID']], duration_col=duration_col, event_col=event_col)
cph.print_summary()

In [ ]:
print(cph.concordance_index_)
su = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
su.columns = ['HR', 'HR lower 95%', 'HR upper 95%', 'pvalue']
su.reset_index().to_csv(f'features/{task_type}features_HR.csv', index=False)
su

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
cph.plot(hazard_ratios=True)
plt.savefig(f'img/{task_type}feature_pvalue.svg')
plt.show()

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

thres = 0.001
bst_split = {'train': 1, 'test_sel':1.13, 'test': 1,}
loc = {'train': 0.75, 'val':0.2, 'test': 0.75,}
for subset, test_data in val_datasets.items():
    c_index = cph.score(test_data[[c for c in test_data.columns if c != 'ID']], scoring_method="concordance_index")
#     y_pred = cph.predict_median(test_data[[c for c in test_data.columns if c != 'ID']])
#     cox_data = pd.concat([test_data, y_pred], axis=1)
#     mean = cox_data.describe()[0.5]['mean']
#     cox_data['HR'] = cox_data[0.5] < mean
    y_pred = cph.predict_partial_hazard(test_data[[c for c in test_data.columns if c != 'ID']])
    cox_data = pd.concat([test_data, y_pred], axis=1)
    mean = cox_data.describe()[0]['50%']
    cox_data['HR'] = cox_data[0] > bst_split[subset] if subset in bst_split else 1
    cox_data['HR'] = cox_data[0] > mean

    dem = (cox_data["HR"] == True)
    results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                           event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
    p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    plt.title(f"Cohort {subset} C-index:{c_index:.3f}")
    plt.ylabel('Probability')
    if sum(dem):
        kmf_high = KaplanMeierFitter()
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label="High Risk")
        kmf_high.plot_survival_function(color='r')
    if sum(~dem):
        kmf_low = KaplanMeierFitter()
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label="Low Risk")
        kmf_low.plot_survival_function(color='g')
    plt.legend(loc=3)
    y_min, y_max = plt.ylim()
    plt.text(0.5, y_min + (y_max-y_min) / 3, f"P{p_value}")
    plt.xlabel('Time(days)')
    add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.show()

# Save predictions

In [ ]:
import os
import numpy as np

def get_prediction(model: CoxPHFitter, data, ID=None, **kwargs):
    hr = model.predict_partial_hazard(data)
    expectation = model.predict_expectation(data)
    
    predictions = pd.concat([hr, expectation], axis=1)
    predictions.columns = ['HR', 'expectation']
    if ID is not None:
        predictions = pd.concat([ID, hr, expectation], axis=1)
        predictions.columns = ['ID', 'HR', 'expectation']
    else:
        predictions = pd.concat([hr, expectation], axis=1)
        predictions.columns = ['HR', 'expectation']
    return predictions

os.makedirs('results', exist_ok=True)
info = []
for subset, test_data in val_datasets.items():
    if subset in get_param_in_cwd('subsets'):
        results = get_prediction(cph, test_data, ID=test_data['ID'])
        results.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)
        results['group'] = subset
        info.append(results)
        pd.merge(results, data[['ID', event_col, duration_col]], 
                 on='ID', how='inner').to_csv(f'features/{task_type}4xtile_{subset}.txt', index=False, sep='\t')
info = pd.concat(info, axis=0)
info.to_csv(f'results/{task_type}cox_predictions.csv', index=False)
info